# Grass Ledger: comfort table build (TabPFN vs baseline)
Runs once in Colab/Kaggle. Output: `comfort_table.json` (tiny lookup the app ships with) and `metrics.json` (numbers for the write-up).

**Only edit the CONFIG cell.** Weather data: Open-Meteo historical API (check their terms and credit them in the post).

In [ ]:
!pip -q install tabpfn pandas requests scikit-learn

## 1. CONFIG (edit this)

In [ ]:
# ---- CONFIG: change these for your town ----
PLACE = "My Town"        # label only
LAT   = 28.61            # CHANGE ME
LON   = 77.21            # CHANGE ME
YEARS_OF_HISTORY = 4     # last 4 years of hourly data
HOLDOUT_MONTHS   = 12    # most recent period held out for the honest comparison
TRAIN_SAMPLE     = 10000 # TabPFN works best on <= ~10k rows; we subsample
SEED = 42
# --------------------------------------------

## 2. Fetch hourly history from Open-Meteo

In [ ]:
import requests, pandas as pd, numpy as np, datetime as dt

end = dt.date.today() - dt.timedelta(days=7)          # archive lags a few days
start = end.replace(year=end.year - YEARS_OF_HISTORY)

r = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": LAT, "longitude": LON,
        "start_date": start.isoformat(), "end_date": end.isoformat(),
        "hourly": "apparent_temperature,precipitation",
        "timezone": "auto",
    },
    timeout=120,
)
r.raise_for_status()
h = r.json()["hourly"]
df = pd.DataFrame({
    "time": pd.to_datetime(h["time"]),
    "apparent_temp": h["apparent_temperature"],
    "precip": h["precipitation"],
}).dropna()

df["month"] = df.time.dt.month
df["hour"]  = df.time.dt.hour
df["doy"]   = df.time.dt.dayofyear
df["rain"]  = (df.precip >= 0.1).astype(int)

df.to_csv("weather_hourly.csv", index=False)   # save the raw CSV for the repo/post
print(PLACE, df.shape, df.time.min(), "->", df.time.max())
df.head()

## 3. Time-based split (hold out the most recent period)

In [ ]:
cutoff = df.time.max() - pd.DateOffset(months=HOLDOUT_MONTHS)
train = df[df.time <= cutoff].copy()
test  = df[df.time >  cutoff].copy()
print("train", len(train), "test", len(test), "cutoff", cutoff)

## 4. Baseline: average by month and hour

In [ ]:
from sklearn.metrics import mean_absolute_error

base_table = train.groupby(["month", "hour"]).apparent_temp.mean()
test["pred_base"] = [base_table.loc[(m, hr)] for m, hr in zip(test.month, test.hour)]
mae_base = mean_absolute_error(test.apparent_temp, test.pred_base)
print(f"Baseline MAE (apparent temp, C): {mae_base:.3f}")

## 5. TabPFN on the same split
TabPFN may ask you to accept a licence / log in to Hugging Face the first time it downloads weights. If so, follow the prompt it prints.

In [ ]:
from tabpfn import TabPFNRegressor

FEATURES = ["month", "hour", "doy"]
tr = train.sample(min(TRAIN_SAMPLE, len(train)), random_state=SEED)

model = TabPFNRegressor()
model.fit(tr[FEATURES].values, tr.apparent_temp.values)

# evaluate on a subsample of the holdout if it is large (keeps Colab fast)
te = test.sample(min(5000, len(test)), random_state=SEED)
pred_tab = model.predict(te[FEATURES].values)
mae_tab = mean_absolute_error(te.apparent_temp, pred_tab)
mae_base_te = mean_absolute_error(te.apparent_temp, te.pred_base)   # baseline on the SAME rows

print(f"Baseline MAE: {mae_base_te:.3f}   TabPFN MAE: {mae_tab:.3f}")
print("TabPFN better" if mae_tab < mae_base_te else "Baseline better or equal (report this honestly)")

## 6. Build the comfort table
Comfort 0-100 from apparent temperature (ideal band 18-27 C, falling to 0 at 5 C and 40 C), reduced by rain probability. Rain probability is the empirical frequency by month and hour from the training data (plain code, no model).

In [ ]:
def comfort_from_temp(t, lo_zero=5, lo_ideal=18, hi_ideal=27, hi_zero=40):
    if lo_ideal <= t <= hi_ideal: return 100.0
    if t < lo_ideal:  return float(np.clip((t - lo_zero) / (lo_ideal - lo_zero), 0, 1) * 100)
    return float(np.clip((hi_zero - t) / (hi_zero - hi_ideal), 0, 1) * 100)

# final model: refit on a sample of ALL history for the shipped table
full = df.sample(min(TRAIN_SAMPLE, len(df)), random_state=SEED)
final = TabPFNRegressor()
final.fit(full[FEATURES].values, full.apparent_temp.values)

rain_p = df.groupby(["month", "hour"]).rain.mean()

grid = pd.DataFrame([(m, hr, dt.date(2025, m, 15).timetuple().tm_yday)
                     for m in range(1, 13) for hr in range(24)],
                    columns=FEATURES)
grid["temp"] = final.predict(grid[FEATURES].values)

table = {}
for _, row in grid.iterrows():
    m, hr = int(row.month), int(row.hour)
    p_rain = float(rain_p.loc[(m, hr)])
    score = comfort_from_temp(row.temp) * (1 - 0.6 * p_rain)
    table.setdefault(str(m), {})[str(hr)] = {
        "temp": round(float(row.temp), 1),
        "rain": round(p_rain, 2),
        "comfort": round(score),
    }
print("example, month 6, hour 14:", table["6"]["14"])

## 7. Export files

In [ ]:
import json, sklearn, tabpfn, importlib.metadata as md

meta = {
    "place": PLACE, "lat": round(LAT, 2), "lon": round(LON, 2),
    "source": "Open-Meteo historical weather API",
    "history_from": str(df.time.min().date()), "history_to": str(df.time.max().date()),
    "generated": str(dt.date.today()),
}
with open("comfort_table.json", "w") as f:
    json.dump({"meta": meta, "table": table}, f, separators=(",", ":"))

metrics = {
    **meta,
    "holdout_months": HOLDOUT_MONTHS,
    "train_rows": int(len(train)), "tabpfn_train_sample": int(len(tr)),
    "holdout_rows_evaluated": int(len(te)),
    "mae_baseline_c": round(float(mae_base_te), 3),
    "mae_tabpfn_c": round(float(mae_tab), 3),
    "tabpfn_version": md.version("tabpfn"),
    "sklearn_version": sklearn.__version__,
}
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))

# Download immediately so a session reset costs nothing
try:
    from google.colab import files
    for name in ["comfort_table.json", "metrics.json", "weather_hourly.csv"]:
        files.download(name)
except ImportError:
    print("Not in Colab: files are in the working directory (Kaggle: Output tab).")